# Sales: Raw -> Bronze

Land the raw CSV extract as-is, only standardizing column names.

In [1]:
%run ../00_config.ipynb
%run ../00_utils.ipynb

/usr/local/lib/python3.12/site-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


[08/23/26 15:10:25] INFO     Using                                                                  ]8;id=7809038;file:///usr/local/lib/python3.12/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=7809039;file:///usr/local/lib/python3.12/site-packages/kedro/framework/project/__init__.py#302\302]8;;\
                             '/usr/local/lib/python3.12/site-packages/kedro/framework/project/rich_                
                             logging.yml' as logging configuration.                                                

[08/23/26 15:10:25] WARNING  /usr/local/lib/python3.12/site-packages/kedro/framework/context/contex ]8;id=7809046;file:///usr/local/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=7809047;file:///usr/local/lib/python3.12/warnings.py#112\112]8;;\
                             t.py:221: UserWarning: Parameters not found in your Kedro project                     
                             config.                                                                               
                             No files of YAML or JSON format found in /app/conf/base or                            
                             /app/conf/local matching the glob pattern(s): ['parameters*',                         
                             'parameters*/**', '**/parameters*']                                                   
                               warn(f"Parameters not found in your Kedro project config.\n{exc!s}")                
                                                                                                                   

                    INFO     No typed parameter requirements found, returning original   ]8;id=7809054;file:///usr/local/lib/python3.12/site-packages/kedro/validation/parameter_validator.py\parameter_validator.py]8;;\:]8;id=7809055;file:///usr/local/lib/python3.12/site-packages/kedro/validation/parameter_validator.py#124\124]8;;\
                             parameters                                                                            

Kedro context loaded from /app
Catalog datasets: ['raw_employees', 'bronze_employees', 'silver_employees', 'gold_employees', 'raw_sales', 'bronze_sales', 'silver_sales', 'gold_sales', 'raw_inventory', 'bronze_inventory', 'silver_inventory', 'gold_inventory', 'parameters']


                    WARNING  /usr/local/lib/python3.12/site-packages/nbformat/validator.py:434:     ]8;id=7809060;file:///usr/local/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=7809061;file:///usr/local/lib/python3.12/warnings.py#112\112]8;;\
                             MissingIDFieldWarning: Cell is missing an id field, this will become a                
                             hard error in future nbformat versions. You may want to use                           
                             `normalize()` on your notebooks before validations (available since                   
                             nbformat 5.1.4). Previous versions of nbformat are fixing this issue                  
                             transparently, and will stop doing so in the future.                                  
                               _validate(nbdict, ref, version, version_minor, relax_add_props)                     
                                                                                                                   

In [2]:
raw_sales = catalog.load("raw_sales")
raw_sales

                    INFO     Loading data from raw_sales (CSVDataset)...                       ]8;id=7809068;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=7809069;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py#1050\1050]8;;\

,Order ID,Customer Name,Product,Quantity,Unit Price,Order Date,Notes
0,1.0,Acme Corp,Widget,10,25.5,2023-01-15,priority
1,2.0,Globex Inc,GADGET,5,12.0,2023-02-20,NaN
2,3.0,Initech LLC,Widget,20,25.5,2023-03-05,bulk order
3,NaN,Bad Row,Unknown,0,0.0,2022-01-01,"bad data, should be dropped"
4,4.0,Umbrella Co,gizmo,8,40.0,2023-04-10,NaN


In [3]:
bronze_sales = standardize_columns(raw_sales)
bronze_sales

,order_id,customer_name,product,quantity,unit_price,order_date,notes
0,1.0,Acme Corp,Widget,10,25.5,2023-01-15,priority
1,2.0,Globex Inc,GADGET,5,12.0,2023-02-20,NaN
2,3.0,Initech LLC,Widget,20,25.5,2023-03-05,bulk order
3,NaN,Bad Row,Unknown,0,0.0,2022-01-01,"bad data, should be dropped"
4,4.0,Umbrella Co,gizmo,8,40.0,2023-04-10,NaN


In [4]:
catalog.save("bronze_sales", bronze_sales)

                    INFO     Saving data to bronze_sales (CSVDataset)...                       ]8;id=7809075;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=7809076;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py#1006\1006]8;;\

## PySpark alternative (reference only)

PySpark isn't installed in this image. Left commented out to show how this
stage would read/write with Spark instead of pandas.

In [5]:
# raw_sales_spark = (
#     spark.read.option("header", "true")
#     .option("inferSchema", "true")
#     .csv(str(PROJECT_ROOT / "data/01_raw/sales.csv"))
# )
#
# bronze_sales_spark = standardize_columns_spark(raw_sales_spark)
#
# bronze_sales_spark.write.mode("overwrite").option("header", "true").csv(
#     str(PROJECT_ROOT / "data/02_bronze/sales.csv")
# )